# Analyzing RTP Streams - Week 6 Discussion Section

## Goal

The goal of this assignment is to introduce you to analyzing RTP (Real-time Transport Protocol) flows using Wireshark. You will learn how to inspect RTP traffic, extract key metrics from packet traces, and use those metrics to infer Quality of Experience (QoE) for real-time applications such as voice and video calls.

**Before you start:** Download the packet capture (pcap) file you will be using for this assignment:
[Download pcap](https://drive.google.com/file/d/1iqNtH8ip3PBdozZ3ODHvZK6851eBcHlv/view)

## Background: What is RTP?

**RTP (Real-time Transport Protocol)** is an application-layer protocol used to deliver audio and video over IP networks. It runs on top of UDP and is the standard transport for real-time media in applications like VoIP, video conferencing (Zoom, Teams, WebRTC), and live streaming.

Key characteristics:
- **Application layer** — RTP sits above the transport layer (UDP) and adds media-specific features such as payload type identification, sequencing, and timestamping.
- **Sequence Numbers** — allow the receiver to detect packet loss and reordering.
- **Timestamps** — allow the receiver to reconstruct the original timing of the media stream (e.g., for smooth playback).
- **SSRC** — a Synchronization Source identifier that uniquely identifies each RTP stream within a session.

### RTP Packet Structure

```
 0                   1                   2                   3
 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|V=2|P|X|  CC   |M|     PT      |       Sequence Number         |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|                           Timestamp                           |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
|           Synchronization Source (SSRC) Identifier            |
+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+
|            Contributing Source (CSRC) Identifiers             |
|                             ....                              |
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+
```

## Why Doesn't Wireshark Automatically Detect RTP?

Because RTP is an **application-layer protocol**, Wireshark cannot reliably identify RTP packets by port number alone. Unlike protocols such as HTTP (port 80) or DNS (port 53), RTP has **no standardized, well-known port**. Applications are free to use any ephemeral UDP port for RTP streams, and those ports are negotiated dynamically (e.g., via SDP in a SIP or WebRTC session). With thousands of possible UDP ports in use, Wireshark would produce too many false positives if it tried to guess.

As a result, we must **force-decode** a UDP flow as RTP. There are two ways to do this:

### Method 1: Wireshark GUI
Right-click on a UDP packet in the flow you want to decode → **Decode As...** → select **RTP** as the protocol. This tells Wireshark to interpret all packets on that port as RTP for the current session.

### Method 2: `tshark` Command-Line (Scripted)
When using `tshark`, pass the `-d` flag to specify which UDP port(s) should be decoded as RTP:

```bash
tshark -r <pcap_name> \
  -d udp.port==<PORT1>,rtp \
  -d udp.port==<PORT2>,rtp \
  -T fields -E separator=, \
  -e frame.time_epoch \
  -e ip.src -e udp.srcport \
  -e ip.dst -e udp.dstport \
  -e rtp.seq -e rtp.timestamp -e rtp.ssrc
```

The `-d udp.port==<PORT>,rtp` flag instructs `tshark` to treat all UDP traffic on that port as RTP, enabling it to parse and export RTP header fields.

## Question 1

Using either method described above (Wireshark GUI or `tshark`), try to identify which flows in thie packet capture are sending RTP.

**Hint:** RTP runs over UDP, so start by identifying UDP flows that carry a large number of packets or bytes — these are good candidates for RTP streams. Once you have a candidate flow, force-decode it as RTP and apply some basic sanity checks to verify you are looking at a real RTP stream:

- **Version field** — Is the RTP version number `2`? (All valid RTP packets must have `V=2`.)
- **Sequence numbers** — Do the sequence numbers increment by 1 with each packet? Gaps may indicate packet loss, but the overall trend should be monotonically increasing.
- **Timestamps** — Do the timestamps increase at a regular rate consistent with the media clock (e.g., 90,000 Hz for video, 8,000 Hz for G.711 audio)?
- **Payload type** — Does the payload type (PT) field correspond to a known codec (e.g., PT 96–127 are dynamic, but should be consistent within a stream)?
- **SSRC consistency** — Does the same SSRC identifier appear across packets in the flow?

**Write down:** Which flows do you observe are using RTP? How many stream identifiers do you observe? How many in each direction?

In [ ]:
# Your Code Goes Here

## RTP Timestamps and Video Frames

In video streaming, a **frame** is a single complete image in the video sequence. A video stream is just a series of frames played back at some frame rate (e.g., 30 fps). Because video frames can be large, a single frame is often split across **multiple RTP packets**. The receiver uses the **RTP timestamp** to know which packets belong to the same frame — all packets belonging to the same frame share the same RTP timestamp.

### How the Receiver Uses the Timestamp

The RTP timestamp does **not** represent wall-clock time directly. Instead, it represents a position on a **media clock** that ticks at a fixed sampling rate:

- **Video** typically uses a clock rate of **90,000 Hz** (90 kHz), meaning the timestamp increments by 90,000 every second of media.
- **Audio** codec sampling rates vary (e.g., 8,000 Hz for G.711, 48,000 Hz for Opus).

The receiver uses consecutive timestamp values to compute the **relative timing** between frames — i.e., how long to wait before displaying the next frame. This is what enables smooth playback.


## Question 2

For each RTP stream you identified in Question 1:

1. **How many unique frames** are sent in the stream? (Recall: packets belonging to the same frame share the same RTP timestamp.)
2. **How many RTP packets** make up each frame? Does this vary across frames?
3. **Plot** the number of packets per frame over time (use the frame's arrival time or frame index on the x-axis, and packet count per frame on the y-axis). Do this for each stream separately.



In [ ]:
# Your Code Goes Here

## Background: Why Do We Need STUN?

Real-time applications like video conferencing often run on devices sitting behind a **NAT (Network Address Translator)**. A NAT maps a device's private IP address (e.g., `192.168.x.x`) to a single public IP address when communicating with the outside world. This creates a problem for peer-to-peer connections: if both peers are behind NATs, neither knows the other's public IP address or which port the NAT has opened for them.

**STUN (Session Traversal Utilities for NAT)** solves this by letting a client ask a well-known STUN server: *"What does my traffic look like from the outside?"* The STUN server reflects back the client's **public (NAT-translated) IP address and port** in its response. The client can then share this public endpoint with the other peer so they can establish a direct connection.

This exchange happens before any RTP media flows, and you can observe it in Wireshark as STUN Binding Request / Binding Response pairs.

---

## Question 3

In Wireshark, filter for STUN traffic (filter: `stun`) and locate a **STUN Binding Response** addressed to your localhost IP.

1. Inspect the payload of the response. Do you observe an IP address embedded in it?
2. What do you think that IP address represents — is it your private local IP, or something else?
3. Why would the STUN server include this address in the response, and what purpose do you think it serves?

In [ ]:
# Your Code Goes Here